# Additional End of week Exercise - week 2

The code below implements a news assistant that retrieves and presents real-time headlines using a news API from "https://newsapi.org". It defines a get_news function that sends a request to an external news service, provides results based on a user-provided query (such as politics, sports, or business), and extracts relevant article titles from the response. The system integrates with a language model that interprets user input, decides when to invoke the news-fetching function, and returns structured output. Tool-calling is handled dynamically, with a fallback mechanism to ensure responses are still generated even if the model fails to trigger the function correctly. Overall, the pipeline enables users to request topic-specific news, fetches the latest articles programmatically, and presents them in a clean, readable format.

In [ ]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
import http.client

In [ ]:
# Initialization

load_dotenv(override=True)

groq_api_key = os.getenv('GROQ_API_KEY')
free_news_api_key = os.getenv('FREE_NEWS_API_KEY')


if groq_api_key:
    print(f"GROQ API Key exists and begins {groq_api_key[:8]}")
else:
    print("GROQ API Key not set")
    
MODEL = "llama-3.3-70b-versatile"

groq_url = "https://api.groq.com/openai/v1"
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

# As an alternative, if you'd like to use Ollama instead of Groq, you can uncomment the following lines and comment out the Groq initialization above. Make sure to have Ollama running locally.
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [ ]:
def get_news(query):#, country):
    conn = http.client.HTTPSConnection("api.freenewsapi.io")

    headers = {
    "x-api-key": free_news_api_key
    }

    # country_map = {
    #     "india": "IN",
    #     "united states": "US",
    #     "usa": "US",
    #     "uk": "GB",
    #     "united kingdom": "GB",
    #     "australia": "AU",
    #     "canada": "CA"
    # }

    # def get_country_code(name):
    #     return country_map.get(name.lower(), "US")

    # # usage
    # country = get_country_code(country)

    conn.request(
        "GET",
        f"/v1/news?published_after=2026-04-14&language=en&country=US&q={query}",
        headers=headers,
    )

    response = conn.getresponse()

    result = response.read().decode()
    data = json.loads(result)
    final=""
    for item in data['data']:
        final += item["title"] + "\n"
        
    conn.close()
    return final

In [ ]:
print(get_news("technology")) #, "US"))

In [ ]:
news_function = {
    "name": "get_news",
    "description": "Get the latest news articles based on specified criteria.",
    "parameters": {
        "type": "object",
        "properties": {
            # "country": {
            #     "type": "string",
            #     "description": "The country code to filter news articles (e.g., 'US' for United States, 'GB' for Great Britain).",
            # },
            "query": {
                "type": "string",
                "description": "A search query to filter news articles (e.g., 'Sports', 'Politics', 'Business').",
            }
        },
        "required": ["query"], #"country"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": news_function}]
tools

In [ ]:
# system_message = """
# You are a news assistant that extracts structured arguments for API calls. Never throw 
# errors or inaccurate news, say news not found please ask for something else.

# Do NOT generate:
# <function=...> or any text-based tool calls

# When the user asks for news:
# - ALWAYS call the function "get_news"
# - NEVER respond with raw text before calling the function
# - ONLY use valid JSON tool calls (no XML, no custom format)

# Rules:
# - query = main topic (e.g., politics, sports, business)
# - country = full country name in lowercase (e.g., india, usa)
# - Keep query short and clean (e.g., "Indian politics" → "politics")
# - Do NOT include country inside query

# After the tool returns results:
# - Summarize the news clearly and concisely in points
# - Don't give this type of message in point give separate news with new line

# """

system_message = """
You are a helpful news assistant.
The user can ask you for the latest news articles on various topics. Your job 
is to extract the relevant information from the user's query and call the 
appropriate tool to fetch the news articles.

If the user does not specify a topic, ask them what kind of news they want.

Do not make up news or give any unnecessary error messages.
If no news is found, say: "No news found, please ask for something else."

List the news articles clearly and concisely, each on a new line.

"""


In [ ]:
#Calling the model in a chat interface without the news articles tool calls to show that the model can respond to user queries but without the real-time news data.
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_news":
            arguments = json.loads(tool_call.function.arguments)
            query = arguments.get('query')
            # country = arguments.get('country')
            news_articles = get_news(query)#,country)
            responses.append({
                "role": "tool",
                "content": news_articles,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
#calling the tool and showing the news articles in the chat interface when the model makes tool calls to get news data. Also added a try-except block to handle cases where the model might not understand the user's query or fails to make a valid tool call. This way, instead of throwing an error, the model will prompt the user to specify what kind of news they want.

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    try:
        response = groq.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

        while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            responses = handle_tool_calls(message)
            messages.append(message)
            messages.extend(responses)
            response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        
        return response.choices[0].message.content
    
    except Exception as e:
        return "Sorry, I didn't understand. What kind of news do you want?"


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()